In [19]:
import pandas as pd
import numpy as np
import os
import re

pd.set_option("display.max_columns", None)

DATA_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"
os.makedirs(PROCESSED_PATH, exist_ok=True)

In [20]:
# Loading all raw files
employee_df = pd.read_csv(f"{DATA_PATH}/employee_attrition.csv")
engagement_df = pd.read_csv(f"{DATA_PATH}/hr_performance_engagement.csv")
occupation_df = pd.read_csv(f"{DATA_PATH}/occupation_data.csv")
essential_skills_df = pd.read_csv(f"{DATA_PATH}/essential_skills.csv")
software_skills_df = pd.read_csv(f"{DATA_PATH}/software_skills.csv")

## 1. Cleaning `employee_attrition.csv`
From notebook 02, I know there are no duplicates and `Age`/`AttritionRisk` are valid. 
Here I need to fix types, parse dates, clean up categories, handle the 319 missing `CustomerSatisfaction` values, and deal with outliers.

**My approach for modeling:**
- **Missing Data:** `CustomerSatisfaction` is 63.8% missing. If I impute it (mean/median), I'm just injecting fake noise that the model might overfit to. Instead of dropping rows and losing data, I'll create a boolean flag (`_missing`). Tree-based models can actually learn from the *fact* that a value is missing.
- **Outliers:** I'm not going to delete outliers. Deleting rows is irreversible and might remove valid edge cases (like highly paid execs). I'll just flag them using IQR so the modeling phase can decide how to handle them.

In [21]:
emp = employee_df.copy()

# 1. Fixing data types
emp["EmployeeID"] = emp["EmployeeID"].astype("int64")
emp["Age"] = emp["Age"].astype("int64")
emp["MonthlySalary"] = pd.to_numeric(emp["MonthlySalary"], errors="raise")
emp["JoiningDate"] = pd.to_datetime(emp["JoiningDate"], format="%Y-%m-%d")
# Using coerce for LastLeaveDate because active employees legitimately don't have an exit date
emp["LastLeaveDate"] = pd.to_datetime(emp["LastLeaveDate"], format="%Y-%m-%d", errors="coerce")

# 2. Cleaning categories: Step 1 - strip whitespace
category_cols = ["Gender", "Department", "JobRole", "Country", "LeaveDayName", "AttritionRisk"]
for c in category_cols:
    emp[c] = emp[c].str.strip()

# 3. Cleaning categories: Step 2 - Inspect actual values before deciding on mappings.
# This gives me the evidence to build my mapping dictionaries.
for c in category_cols:
    print(f"--- {c} ---")
    print(emp[c].value_counts(dropna=False))
    print()

# 4. Cleaning categories: Step 3 - Explicit mapping ONLY where I see real inconsistencies.
# Anything not in the map keeps its original stripped value. Nothing is guessed or dropped.
gender_map = {
    "male": "Male", "m": "Male",
    "female": "Female", "f": "Female",
    "other": "Other", "o": "Other",
}
emp["Gender"] = emp["Gender"].str.lower().map(gender_map).fillna(emp["Gender"])

# 5. Quick sanity check to make sure I didn't break anything or lose rows
assert len(emp) == len(employee_df), "Row count changed!"

# 6. Handling Missing Values: Flagging instead of imputing to prevent data leakage
emp["CustomerSatisfaction_missing"] = emp["CustomerSatisfaction"].isna()

# 7. Handling Outliers: Flagging via IQR. 
for c in ["MonthlySalary", "OvertimeHoursPerMonth", "ProjectsHandled", "TrainingHours"]:
    q1, q3 = emp[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    emp[f"{c}_outlier"] = ~emp[c].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

emp.to_csv(f"{PROCESSED_PATH}/employee_attrition_processed.csv", index=False)
print(f"Saved employee_attrition_processed.csv -> {emp.shape}")

--- Gender ---
Gender
Female    186
Male      165
Other     149
Name: count, dtype: int64

--- Department ---
Department
Sales        99
IT           93
Support      82
Finance      77
HR           77
Marketing    72
Name: count, dtype: int64

--- JobRole ---
JobRole
Account Manager     50
Sales Executive     49
Helpdesk            42
Support Engineer    40
HR Executive        39
Accountant          39
Auditor             38
HR Manager          38
Developer           38
Content Lead        38
SEO Analyst         34
Tester              30
Engineer            25
Name: count, dtype: int64

--- Country ---
Country
Canada     92
India      90
UK         85
France     80
Germany    77
USA        76
Name: count, dtype: int64

--- LeaveDayName ---
LeaveDayName
Friday       91
Saturday     79
Thursday     77
Sunday       66
Tuesday      64
Monday       62
Wednesday    61
Name: count, dtype: int64

--- AttritionRisk ---
AttritionRisk
No     445
Yes     55
Name: count, dtype: int64

Saved employe

## 2. Cleaning `hr_performance_engagement.csv`
Notebook 01 showed 150 exact duplicate rows. I'll drop those, fix types/dates, normalize categories the same way I did above, and rename the join key so it matches the attrition file.

In [22]:
eng = engagement_df.copy()

# 1. Drop exact duplicates and the useless pandas index column
before = len(eng)
eng = eng.drop(columns=["Unnamed: 0"]).drop_duplicates().reset_index(drop=True)
assert before - len(eng) == 150, f"Expected to drop 150 rows, but dropped {before - len(eng)}"
assert eng.duplicated().sum() == 0

# 2. Types / dates
eng["StartDate"] = pd.to_datetime(eng["StartDate"], format="%d-%b-%y")
eng["DOB"] = pd.to_datetime(eng["DOB"], format="%d-%m-%Y")
eng["ExitDate"] = pd.to_datetime(eng["ExitDate"], format="%d-%b-%y", errors="coerce")  # NaT = active

# 3. Align join key with employee_attrition.csv
eng = eng.rename(columns={"Employee ID": "EmployeeID"})

# 4. Categories: strip, inspect, then map
category_cols = ["EmployeeStatus", "EmployeeType", "PayZone", "EmployeeClassificationType",
                 "DepartmentType", "Division", "State", "GenderCode", "RaceDesc",
                 "MaritalDesc", "Performance Score"]
for c in category_cols:
    eng[c] = eng[c].str.strip()

for c in category_cols:
    print(f"--- {c} ---")
    print(eng[c].value_counts(dropna=False))
    print()

# GenderCode is the classic HR export inconsistency (M/F vs Male/Female)
gendercode_map = {
    "male": "Male", "m": "Male",
    "female": "Female", "f": "Female",
}
eng["GenderCode"] = eng["GenderCode"].str.lower().map(gendercode_map).fillna(eng["GenderCode"])

eng.to_csv(f"{PROCESSED_PATH}/engagement_processed.csv", index=False)
print(f"Saved engagement_processed.csv -> {eng.shape}")

--- EmployeeStatus ---
EmployeeStatus
Active                    2458
Voluntarily Terminated     321
Leave of Absence            86
Future Start                69
Terminated for Cause        66
Name: count, dtype: int64

--- EmployeeType ---
EmployeeType
Full-Time    1038
Contract     1008
Part-Time     954
Name: count, dtype: int64

--- PayZone ---
PayZone
Zone A    1062
Zone B     985
Zone C     953
Name: count, dtype: int64

--- EmployeeClassificationType ---
EmployeeClassificationType
Temporary    1033
Full-Time    1013
Part-Time     954
Name: count, dtype: int64

--- DepartmentType ---
DepartmentType
Production              2020
IT/IS                    430
Sales                    331
Software Engineering     115
Admin Offices             80
Executive Office          24
Name: count, dtype: int64

--- Division ---
Division
Field Operations            789
General - Con               509
Engineers                   275
Aerial                      196
Wireline Construction       180
P

## 3. Cleaning `occupation_data.csv`
No missing values or duplicates here. Just stripping whitespace and making absolutely sure `O*NET-SOC Code` is unique. If it's not unique, joining it to the skills tables later will cause a silent many-to-many row explosion.

In [23]:
occ = occupation_df.copy()
for c in occ.columns:
    occ[c] = occ[c].str.strip()

occ = occ.drop_duplicates().reset_index(drop=True)

# Critical check: this is the primary key for the skills tables
assert occ["O*NET-SOC Code"].is_unique, "O*NET-SOC Code must be unique to act as a join key"

occ.to_csv(f"{PROCESSED_PATH}/occupation_master.csv", index=False)
print(f"Saved occupation_master.csv -> {occ.shape}")

Saved occupation_master.csv -> (1016, 3)


## 4. Cleaning `essential_skills.csv`
`Not Relevant` is just a Y-only flag column, so a blank just means "not marked" (not broken data). I'll leave it alone. Just fixing types, whitespace, and parsing the `Date` string so I can do temporal feature engineering later.

In [24]:
ess = essential_skills_df.copy()
for c in ess.select_dtypes(include=["object", "string"]).columns:
    ess[c] = ess[c].str.strip()

# Parsing to datetime (defaults to 1st of the month, which is fine for temporal features)
ess["Date"] = pd.to_datetime(ess["Date"], format="%m/%Y", errors="coerce")

assert ess.duplicated().sum() == 0
assert len(ess) == len(essential_skills_df)

ess.to_csv(f"{PROCESSED_PATH}/essential_skills_processed.csv", index=False)
print(f"Saved essential_skills_processed.csv -> {ess.shape}")

Saved essential_skills_processed.csv -> (18200, 15)


## 5. Cleaning `software_skills.csv` (Skill Name Normalization)
This is where the messy free-text skill names live (`AWS` vs `Amazon Web Services`). If I leave them as-is, the model will treat them as completely different features and dilute the signal. 

I'm using a curated alias map to standardize them. **Important:** I'm keeping the original text and adding a `_clean` column. This way, if I need to add more aliases later, I don't have to re-run the whole pipeline from the raw data, and it's fully auditable.

In [25]:
sw = software_skills_df.copy()
for c in ["Title", "Workplace Example", "Element Name", "Hot Technology", "In Demand"]:
    sw[c] = sw[c].str.strip()

# Sanity check binary flags
assert set(sw["Hot Technology"].unique()) <= {"Y", "N"}
assert set(sw["In Demand"].unique()) <= {"Y", "N"}

# Helper to normalize text for the alias map
def normalize_key(text: str) -> str:
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

# Curated alias table: normalized spelling -> one canonical name.
ALIAS_MAP = {
    "aws": "Amazon Web Services (AWS)", "amazon web services": "Amazon Web Services (AWS)", "aws cloud": "Amazon Web Services (AWS)",
    "ms excel": "Microsoft Excel", "microsoft excel": "Microsoft Excel", "excel": "Microsoft Excel",
    "ms word": "Microsoft Word", "microsoft word": "Microsoft Word",
    "ms powerpoint": "Microsoft PowerPoint", "microsoft powerpoint": "Microsoft PowerPoint",
    "google workspace": "Google Workspace", "g suite": "Google Workspace", "gsuite": "Google Workspace",
    "sql server": "Microsoft SQL Server", "microsoft sql server": "Microsoft SQL Server",
    "mysql": "MySQL", "postgresql": "PostgreSQL", "postgres": "PostgreSQL",
    "salesforce": "Salesforce", "salesforce crm": "Salesforce",
    "sap": "SAP", "sap erp": "SAP",
}

key = sw["Workplace Example"].apply(normalize_key)
sw["WorkplaceExample_clean"] = key.map(ALIAS_MAP).fillna(sw["Workplace Example"])

print(f"Rows remapped by the alias table: {key.isin(ALIAS_MAP).sum()} / {len(sw)}")
print(f"Distinct unmapped values remaining (I can inspect these to extend ALIAS_MAP later): {(~key.isin(ALIAS_MAP)).sum()}")

sw.to_csv(f"{PROCESSED_PATH}/software_skills_processed.csv", index=False)
print(f"Saved software_skills_processed.csv -> {sw.shape}")

Rows remapped by the alias table: 2452 / 31821
Distinct unmapped values remaining (I can inspect these to extend ALIAS_MAP later): 29369
Saved software_skills_processed.csv -> (31821, 8)


## 6. Checking Join Keys & Cardinality
Before I start merging tables in the feature engineering phase, I need to check the relationships. **Important finding:** There is 0% overlap between `employee_attrition.EmployeeID` and `engagement.EmployeeID` — the two tables share no common employee IDs. This means a merge on EmployeeID would produce an empty result, so these tables cannot be joined directly. The occupation and skills tables do share `O*NET-SOC Code` keys, but joining them simultaneously would create a many-to-many explosion, so I need to join them one at a time.

In [26]:
# 1. employee_attrition <-> engagement, via EmployeeID
assert eng["EmployeeID"].notna().all(), "EmployeeID shouldn't be null in engagement data"

not_in_employee = (~eng["EmployeeID"].isin(emp["EmployeeID"])).sum()
print("Engagement EmployeeIDs not found in employee_attrition:", not_in_employee)

emp_is_unique_key = emp["EmployeeID"].is_unique
eng_records_per_employee = eng["EmployeeID"].value_counts()
print("employee_attrition.EmployeeID is unique:", emp_is_unique_key)
print("engagement records per EmployeeID — min/median/max:", 
      eng_records_per_employee.min(), eng_records_per_employee.median(), eng_records_per_employee.max())

# FIX: Correct the conclusion to match the 0% overlap reality
if not_in_employee == len(eng):
    print("-> CRITICAL: 0% overlap. The datasets are completely DISJOINT.")
    print("   They use different ID spaces and CANNOT be merged on EmployeeID.")
    print("   For the ML phase, engagement must be treated as a separate feature set or investigated further.")
else:
    print("-> Relationship is employee_attrition (1) to engagement (many).")
    print("   Future merge MUST use how='left' from the employee side to prevent row explosion.")

# 2. occupation <-> essential_skills / software_skills, via O*NET-SOC Code
assert occ["O*NET-SOC Code"].is_unique  

codes_missing_in_occ_ess = (~ess["O*NET-SOC Code"].isin(occ["O*NET-SOC Code"])).sum()
codes_missing_in_occ_sw = (~sw["O*NET-SOC Code"].isin(occ["O*NET-SOC Code"])).sum()
print("essential_skills codes not found in occupation_master:", codes_missing_in_occ_ess)
print("software_skills codes not found in occupation_master:", codes_missing_in_occ_sw)

ess_per_code = ess["O*NET-SOC Code"].value_counts()
sw_per_code = sw["O*NET-SOC Code"].value_counts()
print("essential_skills rows per O*NET-SOC Code — min/median/max:", 
      ess_per_code.min(), ess_per_code.median(), ess_per_code.max())
print("software_skills rows per O*NET-SOC Code — min/median/max:", 
      sw_per_code.min(), sw_per_code.median(), sw_per_code.max())
print("-> Relationship is occupation (1) to skills (many) in both cases.")
print("   Joining occupation to BOTH skill tables at the same time would create a many-to-many explosion.")
print("   I need to join them separately, one relationship at a time.")

Engagement EmployeeIDs not found in employee_attrition: 3000
employee_attrition.EmployeeID is unique: True
engagement records per EmployeeID — min/median/max: 1 1.0 1
-> CRITICAL: 0% overlap. The datasets are completely DISJOINT.
   They use different ID spaces and CANNOT be merged on EmployeeID.
   For the ML phase, engagement must be treated as a separate feature set or investigated further.
essential_skills codes not found in occupation_master: 0
software_skills codes not found in occupation_master: 0
essential_skills rows per O*NET-SOC Code — min/median/max: 20 20.0 20
software_skills rows per O*NET-SOC Code — min/median/max: 1 21.0 430
-> Relationship is occupation (1) to skills (many) in both cases.
   Joining occupation to BOTH skill tables at the same time would create a many-to-many explosion.
   I need to join them separately, one relationship at a time.


## 7. Final Pipeline Check
Just confirming every output file exists and the row counts match what I expected based on the cleaning steps above.

In [27]:
checks = {
    "employee_attrition_processed.csv": (emp.shape[0], employee_df.shape[0]),
    "engagement_processed.csv": (eng.shape[0], engagement_df.shape[0]),
    "occupation_master.csv": (occ.shape[0], occupation_df.shape[0]),
    "essential_skills_processed.csv": (ess.shape[0], essential_skills_df.shape[0]),
    "software_skills_processed.csv": (sw.shape[0], software_skills_df.shape[0]),
}

for fname in checks:
    path = f"{PROCESSED_PATH}/{fname}"
    print(f"{'OK' if os.path.exists(path) else 'MISSING'}   {fname}")

print("\n--- Row Count Reconciliation ---")
print(f"employee:         {employee_df.shape[0]} -> {emp.shape[0]} (expected: unchanged)")
print(f"engagement:       {engagement_df.shape[0]} -> {eng.shape[0]} (expected: -150 duplicates)")
print(f"occupation:       {occupation_df.shape[0]} -> {occ.shape[0]} (expected: unchanged)")
print(f"essential_skills: {essential_skills_df.shape[0]} -> {ess.shape[0]} (expected: unchanged)")
print(f"software_skills:  {software_skills_df.shape[0]} -> {sw.shape[0]} (expected: unchanged)")

OK   employee_attrition_processed.csv
OK   engagement_processed.csv
OK   occupation_master.csv
OK   essential_skills_processed.csv
OK   software_skills_processed.csv

--- Row Count Reconciliation ---
employee:         500 -> 500 (expected: unchanged)
engagement:       3150 -> 3000 (expected: -150 duplicates)
occupation:       1016 -> 1016 (expected: unchanged)
essential_skills: 18200 -> 18200 (expected: unchanged)
software_skills:  31821 -> 31821 (expected: unchanged)
